# Venturijeva cijev: predvidi → izračunaj → provjeri

Venturijeva cijev pretvara razliku tlakova u procjenu protoka. Ovdje tlak nije savršeno poznat broj, nego rezultat mjerenja. Cilj je odvojiti **modelsku pretpostavku** (stacionaran, nestlačiv tok; poznat koeficijent istjecanja) od **mjerne nesigurnosti**.

## Predvidi

Prije računa odgovori:

1. Ako se izmjerena razlika tlakova učetverostruči, koliko se promijeni protok?
2. Hoće li ista apsolutna pogreška promjera više utjecati na ulazni promjer ili na grlo?
3. Može li diferencijalni senzor sam dokazati da nigdje nema kavitacije?

Posljednje pitanje je važno: ovaj model koristi samo razliku tlakova. Za kavitaciju treba zasebno poznavati **apsolutni** lokalni tlak i tlak para.


## Veza s riješenim primjerom: ulje i živin manometar

Za idealni primjer iz poglavlja 8 uzimamo ulje gustoće 870 kg/m³, živu
gustoće 13600 kg/m³, promjere 60 i 30 mm te razliku razina žive 0,18 m.
Predvidi predznak razlike tlakova prije računa. Ovdje je koeficijent
istjecanja jednak jedinici. Zasebni nastavni slučaj s vodom u nastavku koristi
drugi skup podataka i zadani kalibracijski koeficijent.


## Interaktivni laboratorij

Najprije zapiši predviđanje. Zatim odaberi **Run → Run All Cells** (Pokreni sve ćelije)
u JupyterLiteu ili **Runtime → Run all** u Colabu. Nakon prvog pokretanja mijenjaj
kontrole bez uređivanja koda. Početno učitavanje Pythona može potrajati.

**Istraži** povezuje skicu i grafove; **Provjeri** objašnjava bilance i granice modela;
**Pogledaj kod** prikazuje stvarne računske funkcije. **Spremi A** zadržava slučaj
za usporedbu, a **Početno stanje** vraća odabrani početni slučaj i uklanja usporedbu.
Programske ćelije možeš otvoriti i mijenjati; svi postojeći pokusi i provjere slijede ispod.

Na uskom zaslonu zatvori bočni popis datoteka klikom na ikonu mape.


In [ ]:
# U lokalnom Pythonu i Colabu widgeti su već instalirani.
# Pyodide po potrebi dohvaća istu pinanu inačicu kroz piplite.
try:
    import ipywidgets
except ModuleNotFoundError:
    import piplite
    await piplite.install("ipywidgets==8.1.8")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def venturi_flow(D1, D2, delta_p, rho, Cd):
    """Protok iz diferencijalnog tlaka, SI; Cd je zadana kalibracija."""
    D1, D2, delta_p, rho, Cd = np.broadcast_arrays(D1, D2, delta_p, rho, Cd)
    if not all(np.all(np.isfinite(x)) for x in [D1, D2, delta_p, rho, Cd]):
        raise ValueError('Sve vrijednosti moraju biti konačne.')
    if np.any(D2 <= 0) or np.any(D1 <= D2) or np.any(delta_p < 0):
        raise ValueError('Za ovaj model treba vrijediti D₁ > D₂ > 0 i Δp ≥ 0.')
    if np.any(rho <= 0) or np.any(Cd <= 0) or np.any(Cd > 1):
        raise ValueError('Potrebno je ρ > 0 i 0 < Cd ≤ 1 za ovaj nastavni model.')
    A2 = np.pi*D2**2/4
    beta = D2/D1
    return Cd*A2*np.sqrt(2*delta_p/(rho*(1-beta**4)))


def venturi_state(D1_mm=60., D2_mm=30., dp_kPa=22.478634, rho=870., Cd=1.,
                  uD2_pct=0., udp_pct=0., uCd_pct=0.):
    """Sučelje pretvara mm/kPa u SI; prikazane nesigurnosti su standardne."""
    if not np.all(np.isfinite([uD2_pct, udp_pct, uCd_pct])) or min(uD2_pct, udp_pct, uCd_pct) < 0:
        raise ValueError('Standardne nesigurnosti moraju biti konačne i nenegativne.')
    D1, D2, dp = D1_mm/1000, D2_mm/1000, dp_kPa*1000
    Q = float(venturi_flow(D1, D2, dp, rho, Cd))
    A1, A2 = np.pi*D1**2/4, np.pi*D2**2/4
    v1, v2 = Q/A1, Q/A2
    beta = D2/D1
    terms = np.array([(2*uD2_pct/(1-beta**4)/100)**2,
                      (.5*udp_pct/100)**2, (uCd_pct/100)**2])
    uQ = Q*np.sqrt(terms.sum())
    # Za Cd < 1 idealna energijska razlika nije jednaka izmjerenom Δp.
    dp_ideal = rho*(v2**2-v1**2)/2
    dp_back = .5*rho*(1-beta**4)*(Q/(Cd*A2))**2
    large = max(uD2_pct, udp_pct, uCd_pct) > 2 or beta > .9
    return dict(D1=D1, D2=D2, dp=dp, rho=rho, Cd=Cd, Q=Q, v1=v1, v2=v2,
                uQ=uQ, terms=terms, dp_ideal=dp_ideal, dp_back=dp_back,
                status='limit' if large else 'ok')


In [ ]:
import ipywidgets as widgets
from IPython.display import display
from html import escape
from inspect import getsource


def lab_table(rows):
    """Tekstualni rezultati ostaju dostupni i bez čitanja grafa."""
    return '<table style="width:100%;table-layout:fixed;overflow-wrap:anywhere"><tbody>' + ''.join(
        '<tr><th scope="row" style="text-align:left">' + escape(str(label))
        + '</th><td>' + escape(str(value)) + '</td></tr>'
        for label, value in rows) + '</tbody></table>'


def lab_slider(label, value, minimum, maximum, step):
    return widgets.FloatSlider(
        description=label, value=value, min=minimum, max=maximum, step=step,
        continuous_update=False, readout_format='.4g',
        style={'description_width': '85px'},
        layout=widgets.Layout(width='310px', max_width='100%', min_width='0'))


def make_lab(lab_id, title, controls, presets, calculate, draw, describe, functions):
    """Isti samostalni prikaz u tri bilježnice; račun ne ovisi o widgetima."""
    state = {'busy': False, 'reference': None, 'result': None, 'revision': 0}
    preset = widgets.Dropdown(options=list(presets), description='Slučaj:',
                              layout=widgets.Layout(width='100%', margin='0'))
    save = widgets.Button(description='Spremi A', tooltip='Zapamti trenutačne ulaze za usporedbu')
    clear = widgets.Button(description='Ukloni A')
    reset = widgets.Button(description='Početno stanje')
    summary, checks, comparison = widgets.HTML(), widgets.HTML(), widgets.HTML()
    plot = widgets.Output(layout=widgets.Layout(width='100%', max_width='100%', margin='0'))
    source = '\n\n'.join(getsource(function) for function in functions)
    code = widgets.HTML('<p>Ovo su funkcije koje računaju trenutačni prikaz. '
                        'Možeš ih urediti u prethodnoj programskoj ćeliji i ponovno '
                        'pokrenuti bilježnicu.</p><pre style="white-space:pre-wrap;'
                        'overflow-wrap:anywhere">' + escape(source) + '</pre>')
    tabs = widgets.Tab(children=[plot, checks, code],
                       layout=widgets.Layout(width='100%', min_width='0', margin='0'))
    for index, text in enumerate(['Istraži', 'Provjeri', 'Pogledaj kod']):
        tabs.set_title(index, text)

    def redraw(change=None):
        if state['busy']:
            return
        values = {key: control.value for key, control in controls.items()}
        state['revision'] += 1
        try:
            result = calculate(**values)
        except ValueError as error:
            state['result'] = None
            save.disabled = True
            summary.value = ('<div class="mf1-lab-status" role="status" '
                             'aria-live="polite" data-state="invalid" '
                             f'data-revision="{state["revision"]}"><b>Provjeri ulaze.</b> '
                             + escape(str(error)) + '</div>')
            checks.value = '<p>Za ove ulaze rezultat nije izračunat.</p>'
            with plot:
                plot.clear_output(wait=False)
            return
        state['result'] = result
        save.disabled = False
        overview, verification = describe(result)
        summary.value = (f'<div class="mf1-lab-status" role="status" aria-live="polite" '
                         f'data-state="{result["status"]}" data-revision="{state["revision"]}">'
                         + overview + '</div>')
        checks.value = verification
        reference = state['reference']
        comparison.value = ('<p><b>A:</b> ' + escape(', '.join(
            f'{controls[key].description} {value:g}' for key, value in reference.items()))
            + '. Isprekidana krivulja / zasebni stupci označavaju A.</p>'
            if reference else '<p>Spremi slučaj A pa promijeni ulaze za usporedbu.</p>')
        with plot:
            plot.clear_output(wait=True)
            figures = draw(result, calculate(**reference) if reference else None)
            for figure in figures:
                display(figure)
                plt.close(figure)

    def load_preset(change=None):
        state['busy'] = True
        try:
            for key, value in presets[preset.value].items():
                controls[key].value = value
        finally:
            state['busy'] = False
        redraw()

    def save_reference(button):
        if state['result'] is not None:
            state['reference'] = {key: control.value for key, control in controls.items()}
            redraw()

    def clear_reference(button):
        state['reference'] = None
        redraw()

    def reset_lab(button):
        state['reference'] = None
        load_preset()

    for control in controls.values():
        control.observe(redraw, names='value')
    preset.observe(load_preset, names='value')
    save.on_click(save_reference)
    clear.on_click(clear_reference)
    reset.on_click(reset_lab)
    root = widgets.VBox([
        widgets.HTML('<style>.mf1-lab .widget-html {min-width:0;overflow-wrap:anywhere}'
                     '.mf1-lab .lm-TabBar-tabLabel {white-space:normal!important;overflow-wrap:anywhere;line-height:1.3!important}'
                     '.mf1-lab .lm-TabBar-tab {height:auto!important}'
                     '@media(max-width:600px){.mf1-lab .lm-TabBar {min-height:48px}.mf1-lab .widget-slider {'
                     'display:grid;grid-template-columns:minmax(0,1fr) 65px;height:auto}'
                     '.mf1-lab .widget-slider>.widget-label {grid-column:1/-1;text-align:left}'
                     '.mf1-lab .widget-slider .slider-container {min-width:0}}'
                     '</style><h3>' + escape(title) + '</h3><p>Mijenjaj klizače '
                     'ili klikni broj za točan unos. Račun se obnavlja kad otpustiš klizač.</p>'),
        preset,
        widgets.Box(list(controls.values()), layout=widgets.Layout(flex_flow='row wrap', width='100%')),
        widgets.Box([save, clear, reset], layout=widgets.Layout(flex_flow='row wrap')),
        summary, comparison, tabs], layout=widgets.Layout(width='100%', min_width='0', margin='0'))
    root.add_class('mf1-lab')
    root.add_class('mf1-lab-' + lab_id)
    load_preset()
    display(root)
    return {'root': root, 'controls': controls, 'preset': preset, 'state': state,
            'tabs': tabs, 'save': save, 'clear': clear, 'reset': reset, 'refresh': redraw}


In [ ]:
def draw_venturi(s, reference=None):
    fig, axes = plt.subplots(3, 1, figsize=(5.4, 8.), layout='constrained')
    ax = axes[0]
    for state, color, style, label in [(s, '#256d85', '-', 'Sada'),
                                      (reference, '#8e4519', '--', 'A')]:
        if state is None:
            continue
        x = np.array([0, .25, .43, .57, .9, 1.])
        radius = 500*np.array([state['D1'], state['D1'], state['D2'],
                              state['D2'], state['D1'], state['D1']])
        ax.plot(x, radius, color=color, ls=style, label=label)
        ax.plot(x, -radius, color=color, ls=style)
        if state is s:
            ax.fill_between(x, -radius, radius, color='#7cb5d6', alpha=.25)
            for xpos in [.12, .5, .95]:
                ax.annotate('', xy=(xpos+.035, 0), xytext=(xpos-.035, 0),
                            arrowprops=dict(arrowstyle='->', color='#11202e'))
    ax.set(xlabel='uzdužni položaj (shematski)', ylabel='polumjer (mm)',
           title='Otvoreni prolaz i dva mjerna presjeka')
    ax.axvline(.15, ls=':', color='#536577')
    ax.axvline(.5, ls=':', color='#536577')
    ax.text(.15, 1.02, '1', transform=ax.get_xaxis_transform(), ha='center')
    ax.text(.5, 1.02, '2', transform=ax.get_xaxis_transform(), ha='center')
    ax.legend(fontsize=9)
    ax = axes[1]
    positions = np.array([0., 1.])
    ax.bar(positions-.15, [s['v1'], s['v2']], width=.3, color='#256d85', label='Sada')
    if reference:
        ax.bar(positions+.15, [reference['v1'], reference['v2']], width=.3,
               color='#8e4519', hatch='//', label='A')
    ax.set(xticks=positions, xticklabels=['presjek 1', 'presjek 2'], ylabel='srednja brzina (m/s)')
    ax.legend(fontsize=9)
    ax = axes[2]
    shares = s['terms']/s['terms'].sum()*100 if s['terms'].sum() else np.zeros(3)
    ax.bar(['D₂', 'Δp', 'Cd'], shares, color=['#256d85', '#8e4519', '#765b91'])
    ax.set(ylabel='udio u varijanci Q (%)', ylim=(0, 105), title='Koje mjerenje najviše utječe?')
    for ax in axes:
        ax.grid(axis='y', ls=':', alpha=.35)
    return [fig]


def describe_venturi(s):
    overview = '<b>Protok iz izmjerene razlike tlakova.</b>'
    if s['status'] == 'limit':
        overview += '<p>Osjetljivo područje: provjeri linearnu propagaciju uzorkovanjem. Prag 2 % / β > 0,9 služi upozorenju u ovom pokusu.</p>'
    overview += lab_table([('Protok Q', f'{1000*s["Q"]:.5f} L/s'),
                           ('Standardna nesigurnost u(Q)', f'{1000*s["uQ"]:.5f} L/s'),
                           ('p₁ − p₂', f'{s["dp"]/1000:.5f} kPa'),
                           ('v₁ / v₂', f'{s["v1"]:.3f} / {s["v2"]:.3f} m/s')])
    checks = ('<p>Stacionaran nestlačiv tok. Cd je zadani koeficijent, a shema nije CFD polje. '
              'Tlakovi su poznati samo kao razlika: apsolutni tlak i kavitacija nisu određeni.</p>'
              + lab_table([('Čvorna razlika A₁v₁ − A₂v₂', f'{np.pi*s["D1"]**2/4*s["v1"]-np.pi*s["D2"]**2/4*s["v2"]:.2e} m³/s'),
                           ('Povratni račun Δp', f'{s["dp_back"]/1000:.5f} kPa'),
                           ('Idealna razlika ρ(v₂² − v₁²)/2', f'{s["dp_ideal"]/1000:.5f} kPa')])
              + '<p>Za Cd = 1 vrijedi idealna Bernoullijeva veza. Za Cd &lt; 1 razlika tih '
                'dviju procjena sama ne daje ukupni trajni gubitak kroz cijeli Venturi.</p>'
                '<p>Kontrole u(D₂), u(Δp) i u(Cd) predstavljaju neovisne relativne standardne '
                'nesigurnosti u %. Ostali su ulazi ovdje točno zadani. Linearna procjena '
                'vrijedi za male nesigurnosti; potpuni budžet pet ulaza i Monte Carlo slijede '
                'u ćelijama ispod. Nula znači zanemaren doprinos, ne savršeno mjerenje.</p>')
    return overview, checks


venturi_presets = {
    'Ulje — primjer iz poglavlja 8': dict(D1_mm=60., D2_mm=30., dp_kPa=22.478634,
        rho=870., Cd=1., uD2_pct=0., udp_pct=0., uCd_pct=0.),
    'Voda — zasebni nastavni slučaj': dict(D1_mm=80., D2_mm=40., dp_kPa=18.,
        rho=998., Cd=.985, uD2_pct=.25, udp_pct=100*80/18000, uCd_pct=100*.003/.985),
}
venturi_controls = {
    'D1_mm': lab_slider('D₁ (mm)', 60., 30., 150., 1.),
    'D2_mm': lab_slider('D₂ (mm)', 30., 10., 100., 1.),
    'dp_kPa': lab_slider('Δp (kPa)', 22.478634, 0., 60., .1),
    'rho': lab_slider('ρ (kg/m³)', 870., 300., 1400., 1.),
    'Cd': lab_slider('Cd', 1., .7, 1., .001),
    'uD2_pct': lab_slider('u(D₂) (%)', 0., 0., 5., .05),
    'udp_pct': lab_slider('u(Δp) (%)', 0., 0., 5., .05),
    'uCd_pct': lab_slider('u(Cd) (%)', 0., 0., 5., .05),
}
mf1_lab = make_lab('venturi', 'Venturi: geometrija, protok i mjerenje',
                   venturi_controls, venturi_presets, venturi_state,
                   draw_venturi, describe_venturi, [venturi_flow, venturi_state])


In [ ]:
import numpy as np

rho_oil, rho_hg, g, dh = 870.0, 13600.0, 9.81, 0.18
D1_oil, D2_oil = 0.060, 0.030
dp_oil = (rho_hg-rho_oil)*g*dh
A1_oil, A2_oil = np.pi*D1_oil**2/4, np.pi*D2_oil**2/4
Q_oil = np.sqrt(2*dp_oil/rho_oil/(1/A2_oil**2-1/A1_oil**2))
v1_oil, v2_oil = Q_oil/A1_oil, Q_oil/A2_oil
assert np.isclose(dp_oil, 22478.634, atol=1e-8, rtol=0)
assert np.isclose(1000*Q_oil, 5.2479185043, rtol=1e-10, atol=0)
assert np.isclose(rho_oil*(v2_oil**2-v1_oil**2)/2, dp_oil, rtol=1e-12, atol=0)
print(f"Idealni primjer s uljem: Δp={dp_oil/1000:.3f} kPa; Q={1000*Q_oil:.3f} L/s")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "font.size": 10})


base = dict(D1=0.080, D2=0.040, delta_p=18_000.0, rho=998.0, Cd=0.985)
sigma = dict(D1=0.00015, D2=0.00010, delta_p=80.0, rho=1.0, Cd=0.003)

Q = float(venturi_flow(**base))
A1, A2 = np.pi*base["D1"]**2/4, np.pi*base["D2"]**2/4
v1, v2 = Q/A1, Q/A2
print(f"Q = {1e3*Q:.3f} L/s; v1 = {v1:.3f} m/s; v2 = {v2:.3f} m/s")


## Izračunaj: propagacija senzorske nesigurnosti

Za međusobno neovisne male nesigurnosti prva procjena je

\[
u_Q^2 \approx \sum_i\left(\frac{\partial Q}{\partial x_i}u_{x_i}\right)^2.
\]

Derivacije računamo centriranom razlikom, a rezultat neovisno provjeravamo Monte Carlo uzorkovanjem s fiksnim sjemenom generatora. Slučajni generator ima fiksno sjeme pa je rezultat ponovljiv i u pregledniku.


In [ ]:
def centered_derivative(key, values, scales):
    step = max(scales[key] * 1e-3, abs(values[key]) * 1e-8)
    plus, minus = values.copy(), values.copy()
    plus[key] += step
    minus[key] -= step
    return (float(venturi_flow(**plus)) - float(venturi_flow(**minus))) / (2*step)

gradient = {key: centered_derivative(key, base, sigma) for key in base}
variance_terms = {key: (gradient[key]*sigma[key])**2 for key in base}
u_linear = np.sqrt(sum(variance_terms.values()))

rng = np.random.default_rng(20260801)
n_samples = 50_000
samples = {
    key: rng.normal(base[key], sigma[key], n_samples)
    for key in base
}
Q_mc = venturi_flow(**samples)
u_mc = np.std(Q_mc, ddof=1)
q025, q975 = np.quantile(Q_mc, [0.025, 0.975])

print(f"Linearna standardna nesigurnost: {1e3*u_linear:.4f} L/s")
print(f"Monte Carlo standardna nesigurnost: {1e3*u_mc:.4f} L/s")
print(f"Monte Carlo 95 %-tni interval: [{1e3*q025:.3f}, {1e3*q975:.3f}] L/s")
for key, term in sorted(variance_terms.items(), key=lambda item: -item[1]):
    print(f"  {key:7s}: {100*term/u_linear**2:5.1f} % varijance")


## Provjeri

Tri provjere koriste informacije koje nisu ugrađene u isti numerički korak:

- povratnim modelom izračunavamo tlak iz dobivenog protoka;
- provjeravamo poznati zakon skaliranja \(Q\propto\sqrt{\Delta p}\);
- uspoređujemo linearnu propagaciju s Monte Carlo uzorkovanjem.


In [ ]:
beta = base["D2"]/base["D1"]
delta_p_back = 0.5*base["rho"]*(1-beta**4)*(Q/(base["Cd"]*A2))**2
Q_four_dp = float(venturi_flow(**{**base, "delta_p": 4*base["delta_p"]}))

assert np.isclose(delta_p_back, base["delta_p"], rtol=1e-12)
assert np.isclose(Q_four_dp/Q, 2.0, rtol=1e-12)
assert abs(u_mc/u_linear - 1) < 0.05

labels = list(variance_terms)
shares = np.array([variance_terms[k] for k in labels])/u_linear**2
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(1e3*Q_mc, bins=55, color="#7cb5d6", edgecolor="white")
axes[0].axvline(1e3*Q, color="#b43c35", lw=2, label="nominalno")
axes[0].set(xlabel="Q (L/s)", ylabel="broj uzoraka", title="Propagirana nesigurnost")
axes[0].legend()
axes[1].bar(labels, 100*shares, color="#256d85")
axes[1].set(ylabel="udio u varijanci Q (%)", title="Osjetljivost na ulaze")
axes[1].tick_params(axis="x", rotation=30)
for ax in axes: ax.grid(True, axis="y", ls=":", alpha=.45)
plt.tight_layout(); plt.show()


## Protumači

Ponovi račun s dvostruko većom nesigurnošću \(D_2\). Je li racionalnije poboljšati senzor tlaka ili mjerenje promjera? Zaključak vrijedi samo unutar navedenog modela i raspona; nesigurnost koeficijenta \(C_d\) predstavlja dio kalibracije koji idealna Bernoullijeva jednadžba sama ne može odrediti.
